In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy
import scipy.signal as signal

from util.Processing import discretize


In [ ]:
# to read source
from sklearn.linear_model import ElasticNet
el = ElasticNet()

In [ ]:
# Load real trace data
trace_file_path = 'NaI_trace_filtered_220726_045157_buffer_0.txt'
real_trace = np.loadtxt(trace_file_path, skiprows=1)
# Extract time and voltage columns
real_trace_time = real_trace[:, 0]  # Relative time in µs
real_trace_ADC = real_trace[:, 1]  # ADC amplitudes

# load real listmode data
# Load NaI listmode data from the text file
nai_listmode_file = "NaI_listmode_filtered_220726_045157.txt"  # Use the saved file name
nai_listmode_data = np.loadtxt(nai_listmode_file, skiprows=1)  # Skip header row

# Extract time and energy
nai_listmode_time = nai_listmode_data[:, 0]  # Time in seconds of the day
nai_listmode_energy = nai_listmode_data[:, 1]  # Energy in keV

In [ ]:
mv_per_bit = 1000. / 4096.
tstep = 12e-9  # Sampling rate in seconds (80 MHz)
trace_length = len(real_trace)  # Use real trace length
dt = 1e-9  # Delay parameter
thresh = 5 / mv_per_bit  # Pulse trigger threshold in ADC using mV conversion
baseline = 109 / mv_per_bit  # Estimate baseline from data in ADC using mV conversion
extend = 1
Ch_per_int = 0.499  # got this from Dr. Smith
E_per_ch = 0.251  # keV got this from Dr. Smith
E_per_int = E_per_ch * Ch_per_int  # keV
int_i = 96  # Integration time (samples)
dead_i = int_i  # Dead time

# Load the real pulse data for the kernal
file_path = 'NaI_trace_pulse_220726_045157_buffer_0.txt'
pulse_data = np.loadtxt(file_path, skiprows=1)  # Skip the header row

# Extract time and amplitude
pulsetimes = pulse_data[:, 0]  # Time column (in µs)
pulse = pulse_data[:, 1]  # Amplitude column (normalized)
pulse = pulse[83:170]
# Normalize the pulse
print(np.max(pulse))
pulse -= np.min(pulse)
pulse /= np.max(pulse)  # Ensures the peak is at 1.0
start_to_peak = np.argmax(pulse)

# pulsetimes and pulse are now equivalent to nai_pulse(1.)
nsamples = len(pulse)


# Pulse Integration Method that was used with simulated traces
def nai_trace_to_counts(trace, dt, tstep, thresh, baseline, extend, E_per_int, int_i, dead_i):
    energies = []
    sample_times = []
    n = trace.size
    di = int(dt / tstep)  # Delay in samples
    i = di

    while i < n - dead_i - 1:
        if trace[i] > thresh + baseline:  # Detect pulse above threshold
            # Integrate pulse over int_i samples starting with the first sample above threshold
            clip = trace[i - 5:i - 6 + int_i]
            energy = np.sum(clip - baseline)
            # energy = np.sum(trace[i:i + int_i] - baseline)
            norm_energy = energy * E_per_int  # Convert energy into keV
            energies.append(norm_energy)
            sample_times.append(i)
            i += dead_i  # Apply dead time

            # Paralyzable dead time extension
            if extend > 0:
                while trace[i - 1] > thresh + baseline and i < n - di - extend:
                    i += extend
        else:
            i += 1

    return np.array(energies), np.array(sample_times)

In [ ]:
plt.plot(pulse)
print(pulse.shape, np.argmax(pulse), np.max(pulse), np.min(pulse))

In [ ]:
# Perform pulse integration
energies, event_sample = nai_trace_to_counts(real_trace_ADC, dt, tstep, thresh, baseline, extend, E_per_int, int_i,
                                             dead_i)
event_time = event_sample * tstep * 1e6  # Convert samples to time in microseconds

In [ ]:
# Plot 1: Real Trace
figure = plt.figure(figsize=(12, 4), dpi=300)
ax1 = figure.add_subplot()
ax2 = ax1.twinx()
trace_time = np.arange(len(real_trace)) * tstep * 1e6
ax1.plot(trace_time, real_trace_ADC * mv_per_bit, color='blue', alpha=0.8, label='Real Trace')
ax2.scatter(event_time, energies, color='black', marker='.', s=100, label='Synthetic Pulse Integration (FPGA)')
ax1.set_xlabel('Time (µs)', fontsize=18)
ax1.set_ylabel('mV', fontsize=18)
# ax1.set_ylim(100,140)
ax2.set_ylabel('Energy (keV)', fontsize=18)
plt.title('Real NaI Trace from THOR Observation Campaign', fontsize=20)
# plt.ylim(100,230)
# ax1.set_xlim(10,15)
ax1.set_xlim(0, 290)
# ax1.set_xlim(15, 35)
# ax1.set_xlim(30, 70)
ax2.set_yscale('log')
ax2.set_ylim(10, 10000)
ax2.tick_params(axis='y', labelsize=12)

# Add listmode data to the plot
ax2.scatter(
    nai_listmode_time * 0.95,
    # there is a bug in the timing alignment for the THOR listmode data causing it to be slighly out of alignment with the trace. multiplying by 0.96 fixes the issue (temporarily until I can fix it at the source)
    nai_listmode_energy,
    color='purple',
    marker='x',
    s=125,
    label='NaI Real Listmode Data',
    alpha=1.0
)

plt.legend(loc=2, fontsize=12)
# plt.xlim(40, 60)
plt.xlim(155, 162)
plt.show()

print(pulse.shape, np.argmax(pulse), np.min(pulse))

In [ ]:
figure = plt.figure(figsize=(12, 6), dpi=300)
ax1 = figure.add_subplot()
ax2 = ax1.twinx()
trace_time = np.arange(len(real_trace)) * tstep * 1e6

# def triangle_conv(vec, a=6, slope=1):
#     assert a > 2
#     kernel = np.arange(0, a)
#     if np.mod(a, 2) == 1: #odd
#         kernel[a//2:] = kernel[:a//2+1][::-1]
#     else: #even
#         kernel = kernel
#     print(kernel)
#     return np.convolve(vec, slope * kernel, mode='same') / slope

volts = real_trace_ADC * mv_per_bit - baseline * mv_per_bit
ax1.plot(trace_time, volts, color='b', marker='', linestyle='-', alpha=.8, label='V', zorder=2)
peaks, props = signal.find_peaks(volts)
ax1.plot(trace_time[peaks], volts[peaks], color='blue', marker='*', linestyle='', label='V peaks', zorder=2)

# # Peaks of derivative
# dv = np.diff(volts)
# n = 10
# dv_smoothed = scipy.ndimage.gaussian_filter1d(n * dv, sigma=2)
# # dv_smoothed = scipy.ndimage.gaussian_filter1d(n * dv, sigma=.75)
# dv_smoothed /= n
# # dv_smoothed = triangle_conv(dv, a=5, slope=10)
# peaks, props = signal.find_peaks(np.abs(dv_smoothed), prominence=.25, width=.05)
# peaks, props = signal.find_peaks(dv_smoothed, prominence=None, width=None, plateau_size=1.5)
# # ax2.plot(trace_time[:-1], dv, color='green', alpha=.1, zorder=1)
# ax2.plot(trace_time[:-1], dv_smoothed, color='green', marker='', linestyle='-', alpha=.5, label='dV Smoothed', zorder=1)
# ax2.plot(trace_time[peaks], dv_smoothed[peaks], color='green', marker='*', linestyle='', label='dV peaks', zorder=2)
# ax1.plot(trace_time[peaks], volts[peaks], color='green', marker='*', linestyle='', label='dV peaks', zorder=2)
# 
# extrema = signal.argrelextrema(dv_smoothed, np.greater, order=3, mode='clip')
# ax2.plot(trace_time[extrema], dv_smoothed[extrema], color='green', marker='*', linestyle='', label='dV peaks', zorder=2)

# # Second Derivative: derivative of smoothed first derivative
# dv = np.diff(volts)
# n = 10
# dv_smoothed = scipy.ndimage.gaussian_filter1d(n * dv, sigma=5)
# dv_smoothed /= n
# d2v2v = np.diff(dv_smoothed)
# zero_crossings = np.where(np.sign(d2v2v[:-1]) * np.sign(d2v2v[1:]) < 0)[0]
# 
# peaks, props = signal.find_peaks(dv_smoothed, prominence=None, width=None, plateau_size=1.5)
# # ax2.plot(trace_time[:-1], dv, color='green', alpha=.1, zorder=1)
# ax2.plot(trace_time[:-1], dv_smoothed, color='green', marker='', linestyle='-', alpha=.5, label='dV Smoothed', zorder=1)
# ax2.plot(trace_time[peaks], dv_smoothed[peaks], color='green', marker='*', linestyle='', label='dV peaks', zorder=2)
# ax2.plot(trace_time[:-2], d2v2v, color='cyan', alpha=.5, label='d2V2', zorder=1)
# ax2.plot(trace_time[zero_crossings], 5 * d2v2v[zero_crossings], color='cyan', marker='*', linestyle='', label='d2V2 Zero crossings', zorder=2)
# ax2.plot(trace_time[zero_crossings+1], 5 * d2v2v[zero_crossings+1], color='red', marker='*', linestyle='', label='d2V2 Zero crossings', zorder=2)
# ax1.plot(trace_time[peaks], volts[peaks], color='green', marker='*', linestyle='', label='dV peaks', zorder=2)

# Second derivative peaks
# d2v2v = np.diff(np.diff(volts))
# d2v2_smoothed = scipy.ndimage.gaussian_filter1d(d2v2v, sigma=2)
# peaks, props = signal.find_peaks(np.abs(d2v2_smoothed), prominence=None, width=1)
# ax2.plot(trace_time[:-2], 5 * d2v2_smoothed, color='cyan', alpha=.5, label='d2V2 Smoothed', zorder=1)
# ax2.plot(trace_time[peaks], 5 * d2v2_smoothed[peaks], color='cyan', marker='*', linestyle='', label='d2V2 peaks', zorder=2)
# ax1.plot(trace_time[peaks], volts[peaks], color='green', marker='*', linestyle='', label='dV peaks', zorder=2)

# # Zero crossings in the second derivative
# d2v2v = np.diff(np.diff(volts))
# d2v2_smoothed = scipy.ndimage.gaussian_filter1d(d2v2v, sigma=2)
# zero_crossings = np.where(np.sign(d2v2_smoothed[:-1]) * np.sign(d2v2_smoothed[1:]) < 0)[0] # note should be the index before the corss rather than after...
# ax2.plot(trace_time[:-2], 5 * d2v2_smoothed, color='cyan', alpha=.5, label='d2V2 Smoothed', zorder=1)
# ax2.plot(trace_time[zero_crossings], 5 * d2v2_smoothed[zero_crossings], color='cyan', marker='*', linestyle='', label='d2V2 Zero crossings', zorder=2)
# # ax2.plot(trace_time[zero_crossings+1], 5 * d2v2_smoothed[zero_crossings+1], color='red', marker='*', linestyle='', label='d2V2 Zero crossings', zorder=2)
# ax1.plot(trace_time[zero_crossings+1], volts[zero_crossings+1], color='cyan', marker='*', linestyle='', alpha=.5, label='d2V2 Zero crossings', zorder=2)

# ax1.plot([50.75] * 2, [0, 600], 'r--', alpha=.1)

plt.xlim(155, 162)
# plt.xlim(50, 52)
# ax1.set_ylim(-1, 600)
ax1.legend(loc=2, fontsize=12)
ax2.legend(loc=1, fontsize=12)
plt.show()

# TODO README: Would be nice to be able to find events that just generate inflction points, like around 50.75 microS, but this is not realistic without a math based approach
# Direct peak finding is probably as good as it will get...

In [ ]:
slice_mask = np.logical_and(155 <= trace_time, trace_time <= 162)
time_slice = trace_time[slice_mask]
volts_slice = volts[slice_mask]
peaks, props = signal.find_peaks(volts_slice)
print(peaks.size)

deconv = np.zeros_like(volts_slice)
adjusted_listmode = np.zeros_like(peaks)
start_to_peak = np.argmax(pulse)

n = 12
fig, axes = plt.subplots(n,1, figsize=(8, 20), dpi=200)
for i in range(n):
    here = peaks[i]
    prior_deconv = deconv[here].copy()
    adjusted_magnitude = volts_slice[here] - prior_deconv
    adjusted_listmode[i] = adjusted_magnitude
    deconv[here - start_to_peak: here - start_to_peak + pulse.size] += discretize(adjusted_magnitude * pulse, bits=8)
    print(('Iteration: {}, Peak: {:.2f}, Trace Mag i: {:.0f},  Prior Deconv Mag: {:.0f},' +
          'Adjusted Mag: {:.0f}, Post Deconv Mag: {:.0f}').format(i, time_slice[here], volts_slice[here], prior_deconv, adjusted_magnitude, deconv[here]))

    axes[i].plot(time_slice, volts_slice, color='blue', marker='', linestyle='-', label='V', alpha=1, zorder=0)
    axes[i].plot(time_slice[peaks], volts_slice[peaks], color='blue', marker='.', markersize=5, linestyle='', label='V peaks', zorder=2)
    axes[i].plot(time_slice[peaks[:i]], adjusted_listmode[:i], color='red', marker='.', markersize=5, linestyle='', label='Adjusted Magnitude', zorder=3)

    # Current Deconv
    for j in range(i):
        start = np.min(time_slice)
        axes[i].plot(time_slice[peaks[j] - start_to_peak: peaks[j] - start_to_peak + pulse.size],
                     adjusted_listmode[j] * pulse,
                     color='green', marker='', markersize=0, linestyle='-', alpha=.3, label='Current Deconv Estimate', zorder=1)

    index = np.arange(here - start_to_peak + pulse.size)

plt.savefig('Massive_Visualization.png')


In [ ]:
n = 2
fig, axes = plt.subplots(n,1, figsize=(10, 6), dpi=200)

rem = volts_slice.copy()
peaks, props = signal.find_peaks(rem, prominence=5)
new_peaks = peaks.copy()

for iter in range(n):
    if n == 1:
        axes = [axes]
        
    peaks = np.unique(np.concatenate((peaks, new_peaks)))
    
    updated_listmode = np.zeros_like(peaks)
    deconv = np.zeros_like(time_slice)
    
    print('Peaks', new_peaks)
    print('Times:', time_slice[new_peaks])
    
    for i, peak in enumerate(peaks):
        adjusted_magnitude = volts_slice[peak] - deconv[peak]
        updated_listmode[i] = adjusted_magnitude
        start = peak - start_to_peak
        deconv[start: start + pulse.size] += adjusted_magnitude * pulse
    deconv = discretize(deconv, bits=8) # discretization is non-physical and should be applied after sum
        
    axes[iter].plot(time_slice, np.zeros_like(time_slice), 'red', alpha=.05)
        
    axes[iter].plot(time_slice, volts_slice, color='blue', marker='', linestyle='-', label='V', alpha=1, zorder=0)
    axes[iter].plot(time_slice[peaks], rem[peaks], color='blue', marker='.', markersize=5, linestyle='', label='Prior Remainder', zorder=2)
    axes[iter].plot(time_slice[peaks], updated_listmode, color='red', marker='.', markersize=5, linestyle='', label='Adjusted Magnitude', zorder=3)
    
    # Trace minus Deconv iteration
    rem = scipy.ndimage.gaussian_filter1d(volts_slice - deconv, sigma=2)
    new_peaks, props = signal.find_peaks(rem, prominence=5)
    print('New Peaks', new_peaks)
    print('New Times:', time_slice[new_peaks])
    
    axes[iter].plot(time_slice, rem, color='green', marker='', linestyle='-', label='Remainder', alpha=.2, zorder=0)
    axes[iter].plot(time_slice[new_peaks], rem[new_peaks], color='green', marker='*', linestyle='', label='Remainder Peaks', alpha=.2, zorder=0)
    axes[iter].legend(loc=1)
    
    temp = np.unique(np.concatenate((peaks, new_peaks)))
    print('Combined Peaks', new_peaks)
    print('Combined Diffs (<{} will degrade the "causal forward only" assumption) {}'.format(start_to_peak, np.diff(new_peaks)))
    print('Combined Times:', time_slice[new_peaks])


In [ ]:
# Discussion points: This works ok. Peaks are better than integration because we can better account for the mutual interaction between events. We avoid using most knowledge of the shape of the response. This might be improved marginally by doing it iteratively and subtracting out the estimate deconv from the trace. Maybe we can find inflection point photons
# This also has the nice property of being robust to misshapen kernels and discretization!

# Should non-linearities in the scintillator not matter? Charge is basically created instantaneously by the scintillator/photomultiplier so that non-linearity is already account for in deposited charge. This would indicate problems we see stem from non-linearities in amplification, discretization, or bad kernel shape.

# There is no avoiding sampling based methods at high countrates. I am quite certain. Also, deconvolution is still valid for "coincident" photons. The problem lies in
# treating the deconvolved signal as a spectrum, which is no different than the existing listmode.
# - Counterpoint: the fact that pulse shap discrimination is even possible suggests that charge is not in fact instantanesouly deposited...


In [ ]:
# Crazy idea: 